# Fake News Classification with BiLSTM — Dataset 1

This notebook uses the Fake or Real News dataset. The original train/test files are kept separate: validation data is split from the training file, and the test file is reserved for final evaluation.

**Target:** `label` (`FAKE` / `REAL`)  
**Input text:** `text`  
**Model:** stacked Bidirectional LSTM

In [ ]:
from pathlib import Path
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
import nltk

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dropout, Dense, Bidirectional, LSTM
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Available GPUs:", len(tf.config.list_physical_devices("GPU")))

## 1. Load the dataset

In [ ]:
DATA_DIR_CANDIDATES = [
    Path("../data/dataset_1_fake_or_real_news"),
    Path("data/dataset_1_fake_or_real_news"),
]
DATA_DIR = next((p.resolve() for p in DATA_DIR_CANDIDATES if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset 1 folder not found. Start Jupyter from the repository root "
        "or from the notebooks directory."
    )

train_df = pd.read_csv(DATA_DIR / "train.csv", sep=";")
test_df = pd.read_csv(DATA_DIR / "test.csv", sep=";")

required_columns = {"text", "label"}
for name, frame in [("train", train_df), ("test", test_df)]:
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f"{name}.csv is missing required columns: {sorted(missing)}")

train_df = train_df.dropna(subset=["text", "label"]).copy()
test_df = test_df.dropna(subset=["text", "label"]).copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Columns:", train_df.columns.tolist())
display(train_df.head())

## 2. Explore and prepare labels

In [ ]:
print("Training label distribution:")
print(train_df["label"].value_counts())

sns.countplot(data=train_df, x="label", order=["FAKE", "REAL"])
plt.title("Training data: fake vs real")
plt.xlabel("News label")
plt.ylabel("Count")
plt.show()

label_encoder = LabelEncoder()
y_train_full = label_encoder.fit_transform(train_df["label"].astype(str))
known_labels = set(label_encoder.classes_)
if not set(test_df["label"].astype(str)).issubset(known_labels):
    raise ValueError("The test set contains labels not present in the training set.")

y_test = label_encoder.transform(test_df["label"].astype(str))
print("Label mapping:", dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

## 3. Text preprocessing and data split

The test file is not used to fit the tokenizer or tune the model. A validation split is created only from the training file.

In [ ]:
nltk.download("stopwords", quiet=True)
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = re.sub(r"[^a-zA-Z]", " ", str(text)).lower()
    tokens = [
        stemmer.stem(word)
        for word in text.split()
        if word not in stop_words
    ]
    return " ".join(tokens)

X_train_full_text = train_df["text"].astype(str).map(preprocess_text)
X_test_text = test_df["text"].astype(str).map(preprocess_text)

X_train_text, X_val_text, y_train, y_val = train_test_split(
    X_train_full_text,
    y_train_full,
    test_size=0.1765,
    random_state=SEED,
    stratify=y_train_full,
)

print("Train:", len(X_train_text), "Validation:", len(X_val_text), "Test:", len(X_test_text))

## 4. Tokenization and padding

The vocabulary is learned from the training subset only. Unknown words are mapped to an out-of-vocabulary token.

In [ ]:
MAX_VOCAB_SIZE = 10000
MAX_SEQUENCE_LENGTH = 256
EMBEDDING_DIM = 100

tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

def encode_texts(texts):
    sequences = tokenizer.texts_to_sequences(texts)
    return pad_sequences(
        sequences,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post",
    )

X_train = encode_texts(X_train_text)
X_val = encode_texts(X_val_text)
X_test = encode_texts(X_test_text)

print("Encoded train shape:", X_train.shape)
print("Vocabulary size used:", min(len(tokenizer.word_index) + 1, MAX_VOCAB_SIZE))

## 5. Build and train the BiLSTM model

In [ ]:
model = Sequential([
    Embedding(
        input_dim=MAX_VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH,
    ),
    Dropout(0.2),
    Bidirectional(LSTM(128, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(256, return_sequences=True)),
    Dropout(0.2),
    Bidirectional(LSTM(128)),
    Dense(1, activation="sigmoid"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adamax(),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)
model.summary()

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=200,
    callbacks=[early_stopping],
    verbose=1,
)

## 6. Training curves

In [ ]:
history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history_df["accuracy"], label="Train")
axes[0].plot(history_df["val_accuracy"], label="Validation")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_df["loss"], label="Train")
axes[1].plot(history_df["val_loss"], label="Validation")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Final test-set evaluation

The test set is evaluated after training and is not used for early stopping or model selection.

In [ ]:
test_probabilities = model.predict(X_test, verbose=0).ravel()
y_pred = (test_probabilities >= 0.5).astype(int)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred, zero_division=0))
print("F1-score :", f1_score(y_test, y_pred, zero_division=0))
print()
print(classification_report(
    y_test,
    y_pred,
    target_names=[str(label) for label in label_encoder.classes_],
    zero_division=0,
))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=label_encoder.classes_,
    yticklabels=label_encoder.classes_,
)
plt.title("Confusion matrix — Dataset 1")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Notes

- Results may differ from the historical notebook because this version keeps the supplied test split untouched and uses a separate validation subset.
- Training a stacked BiLSTM can be time-consuming on CPU.
- Run the notebook from the repository root or from the `notebooks/` directory.